# 🎓 Day 1 · Session 5
# 05A · RAG Fundamentals
## Why RAG exists, how embeddings retrieve meaning, and how grounding works

> **Teaching flow:** Why → What → How → When to use

## 🎯 Learning objectives

- Understand the concept clearly
- Run a simple, reliable demonstration
- Connect the code to the RAG architecture shown in the slides
- Recognise limitations and production considerations

## 🔧 Setup

Create a `.env` file in the notebook folder:

```env
OPENAI_API_KEY=your-key-here
```

The notebook also has an offline fallback so every cell can execute without an API key.

In [ ]:
# Run once if packages are missing
# %pip install -q -U openai python-dotenv numpy pandas scikit-learn

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

load_dotenv(Path.cwd() / '.env', override=False)
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key) if (OpenAI is not None and api_key) else None

CHAT_MODEL = os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')
EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')

if OpenAI is None:
    print('OpenAI package not installed. Run the installation cell for live API mode.')
print('OpenAI mode:', 'LIVE API' if client else 'OFFLINE FALLBACK')
print('Chat model:', CHAT_MODEL)
print('Embedding model:', EMBEDDING_MODEL)


In [ ]:
def generate_answer(prompt, instructions='You are a helpful academic assistant.'):
    if client is None:
        return '[Offline mode] Add OPENAI_API_KEY to run the live LLM answer.'
    response = client.responses.create(
        model=CHAT_MODEL,
        instructions=instructions,
        input=prompt,
        max_output_tokens=500,
    )
    return response.output_text

def openai_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return np.array(response.data[0].embedding, dtype=float)

def cosine_score(a, b):
    a, b = np.asarray(a), np.asarray(b)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denominator) if denominator else 0.0

## 1️⃣ Why RAG?

An LLM does not automatically know your private or newly updated documents. Without trusted context it may give a generic or plausible-sounding answer.

> **RAG = Retrieve relevant knowledge + give it to the LLM + generate a grounded answer.**

In [ ]:
question = 'What is the M.Tech CSE fee at ABC Institute of Technology?'
print('Question:', question)
print()
print('Answer without institutional context:')
print(generate_answer(question))


## 2️⃣ Grounding with trusted context

In [ ]:
college_handbook = '''
ABC Institute of Technology - Department Handbook

M.Tech CSE Fee Structure:
The tuition fee for M.Tech CSE is Rs. 50,000 per semester.
The programme duration is 4 semesters.
GATE-qualified students are eligible for a 50% tuition scholarship.

NLP Elective:
The NLP elective is offered in Semester 3.
Prerequisites: Machine Learning and Python Programming.
Faculty Coordinator: Dr. Kavitha Raman.

Attendance Rule:
Students must maintain 75% attendance to be eligible for final examinations.
'''

def grounded_prompt(question, context):
    return f'''Use ONLY the supplied context.
If the answer is absent, say: "I don't have this information in the provided knowledge base."
Do not guess.

CONTEXT:
{context}

QUESTION:
{question}
'''

q = 'What is the M.Tech CSE fee?'
print(generate_answer(grounded_prompt(q, college_handbook)))

## 3️⃣ Semantic search: meaning, not exact words

The query and document can use different words but still express the same meaning.

In [ ]:
documents = [
    'M.Tech CSE tuition fee is Rs. 50,000 per semester.',
    'The NLP elective requires Machine Learning and Python Programming.',
    'Students must maintain 75% attendance.',
    'The campus library is open from 8 AM to 8 PM.',
    'The AI lab has GPU workstations for deep learning experiments.'
]
query = 'cost of masters programme'

if client:
    query_vector = openai_embedding(query)
    document_vectors = [openai_embedding(d) for d in documents]
    scores = [cosine_score(query_vector, v) for v in document_vectors]
    mode = 'OpenAI embeddings'
else:
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(documents + [query])
    scores = sklearn_cosine_similarity(matrix[-1], matrix[:-1]).flatten()
    mode = 'offline TF-IDF fallback'

result = pd.DataFrame({'document': documents, 'similarity': scores}).sort_values('similarity', ascending=False)
print('Retrieval mode:', mode)
result

## 4️⃣ Chunking

Large documents are split into smaller retrievable pieces. A practical starting point is **400–600 characters with 50–100 characters overlap**, then tune using evaluation.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    if chunk_size <= 0:
        raise ValueError('chunk_size must be positive')
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError('overlap must be >= 0 and smaller than chunk_size')
    cleaned = text.strip()
    chunks = []
    start = 0
    while start < len(cleaned):
        end = min(start + chunk_size, len(cleaned))
        chunks.append(cleaned[start:end])
        if end == len(cleaned):
            break
        start = end - overlap
    return chunks

chunks = chunk_text(college_handbook, chunk_size=250, overlap=50)
for i, item in enumerate(chunks, 1):
    print(f'--- Chunk {i} ---')
    print(item, '\n')

## 5️⃣ Two phases of RAG

**Indexing — done when documents change:** load → clean → chunk → embed → store.

**Retrieval — done for every question:** embed question → find relevant chunks → build grounded prompt → generate answer.

## ⭐ Wow moment

Ask the same question twice: first without the handbook and then with the handbook. The model itself has not changed—the **information supplied at query time** has changed.

## 🧪 Mini lab

Try: NLP prerequisites, faculty coordinator, attendance requirement, and hostel fee. The hostel question should trigger graceful refusal.

In [ ]:
questions = [
    'What are the prerequisites for NLP?',
    'Who coordinates NLP?',
    'What is the attendance requirement?',
    'What is the hostel fee?'
]
for item in questions:
    print()
    print('QUESTION:', item)
    print('ANSWER:', generate_answer(grounded_prompt(item, college_handbook)))


## ✅ Takeaways

- RAG gives an LLM access to trusted knowledge at query time.
- Embeddings support semantic retrieval.
- Chunking quality directly affects retrieval quality.
- Grounding instructions must allow the model to say **I don't know**.
- RAG improves grounding, but does not guarantee correctness; retrieval and answers must be evaluated.